# 全球 ETF 宏观条件收益研究

本 Notebook 使用本地快照研究 SPY、TLT、GLD 在利率和实际利率状态下的条件收益。

## 使用说明

执行后请根据最后的条件收益表填写结论。本 Notebook 不生成交易指令，也不会自动修改 A 股评分器。

## 方法与边界

- 少量 ETF 不进行 5 组或 10 组横截面分组，改用宏观状态条件收益。
- 日频利率先聚合到月末；资产收益使用月度调整收盘价变化。
- 所有数据同时满足 observation_date <= target_date 和 available_at <= target_date，避免未来函数。
- TLT 的利率代理优先使用 DGS30；缺少时回退 DGS10，并标记精度降级。

### 默认阈值

利率月度变化 >= +0.20 个百分点定义为加息/利率上行，<= -0.20 个百分点定义为降息/利率下行；实际利率变化阈值为 ±0.10 个百分点。

## 数据

读取 `data/processed/global_macro/*.csv`。请先运行 `fetch_global_macro_data.py`。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# 兼容从 qteasy_lab 根目录或 notebooks/ 目录运行：定位 data/processed/global_macro
CANDIDATES = [
    Path.cwd() / 'data' / 'processed' / 'global_macro',
    Path.cwd().parent / 'data' / 'processed' / 'global_macro',
]
DATA_DIR = next((p for p in CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'未找到本地宏观数据目录：{CANDIDATES}')
TARGET_DATE = pd.Timestamp.today().normalize()
ASSETS = ['SPY', 'TLT', 'GLD']
print('数据目录：', DATA_DIR.resolve())
print('目标日期：', TARGET_DATE.date())

数据目录： D:\Project DPS\research_engines\qteasy_lab\data\processed\global_macro
目标日期： 2026-08-04


In [2]:
def load_series(series_id: str) -> pd.DataFrame:
    path = DATA_DIR / f'{series_id}.csv'
    if not path.exists():
        return pd.DataFrame(columns=['series_id', 'observation_date', 'available_at', 'value', 'quality_level'])
    frame = pd.read_csv(path)
    frame['observation_date'] = pd.to_datetime(frame['observation_date'], errors='coerce')
    frame['available_at'] = pd.to_datetime(frame['available_at'], errors='coerce')
    frame['value'] = pd.to_numeric(frame['value'], errors='coerce')
    frame = frame.dropna(subset=['observation_date', 'available_at', 'value'])
    frame = frame.loc[(frame['observation_date'] <= TARGET_DATE) & (frame['available_at'] <= TARGET_DATE)]
    return frame.sort_values('observation_date')

macro = {series_id: load_series(series_id) for series_id in ['DGS10', 'DGS2', 'DGS30', 'DFII10']}
asset_prices = {series_id: load_series(series_id) for series_id in ASSETS}
print({key: len(value) for key, value in {**macro, **asset_prices}.items()})

{'DGS10': 5899, 'DGS2': 5899, 'DGS30': 5899, 'DFII10': 5899, 'SPY': 5932, 'TLT': 5932, 'GLD': 5458}


## 月度宏观状态

按月末聚合利率与 ETF 价格，并建立利率状态。

In [3]:
def month_end(frame: pd.DataFrame) -> pd.Series:
    if frame.empty:
        return pd.Series(dtype='float64')
    return frame.set_index('observation_date')['value'].resample('ME').last()

rate10 = month_end(macro['DGS10'])
rate2 = month_end(macro['DGS2'])
rate30 = month_end(macro['DGS30'])
real10 = month_end(macro['DFII10'])
rate_level = rate30 if not rate30.empty else rate10
rate_proxy = 'DGS30' if not rate30.empty else 'DGS10'
state_frame = pd.DataFrame({'rate': rate_level, 'real_rate': real10})
state_frame['spread_10y2y'] = rate10 - rate2
state_frame['rate_change'] = state_frame['rate'].diff()
state_frame['real_rate_change'] = state_frame['real_rate'].diff()
state_frame['rate_state'] = np.select(
    [state_frame['rate_change'] >= 0.20, state_frame['rate_change'] <= -0.20],
    ['加息/利率上行', '降息/利率下行'], default='利率平稳'
)
state_frame['term_state'] = np.where(state_frame['spread_10y2y'] < 0, '期限结构倒挂', '期限结构正常')
state_frame['real_rate_state'] = np.select(
    [state_frame['real_rate_change'] >= 0.10, state_frame['real_rate_change'] <= -0.10],
    ['实际利率上行', '实际利率下行'], default='实际利率平稳'
)
state_frame.tail()

,rate,real_rate,spread_10y2y,rate_change,real_rate_change,rate_state,term_state,real_rate_state
observation_date,,,,,,,,
2026-03-31,4.88,2.00,0.51,0.24,0.28,加息/利率上行,期限结构正常,实际利率上行
2026-04-30,4.98,1.94,0.52,0.10,-0.06,利率平稳,期限结构正常,实际利率平稳
2026-05-31,4.99,2.07,0.47,0.01,0.13,利率平稳,期限结构正常,实际利率上行
2026-06-30,4.91,2.20,0.30,-0.08,0.13,利率平稳,期限结构正常,实际利率上行
2026-07-31,5.27,2.47,0.47,0.36,0.27,加息/利率上行,期限结构正常,实际利率上行


## 条件收益表

输出各资产在不同宏观状态下的月均收益、波动、最大回撤和胜率。

In [4]:
def asset_monthly_returns(series_id: str) -> pd.Series:
    return month_end(asset_prices[series_id]).pct_change().rename(series_id)

def conditional_stats(asset: str, state_name: str, state_series: pd.Series, returns: pd.Series) -> dict:
    frame = pd.concat([state_series.rename('state'), returns.rename('return')], axis=1).dropna()
    frame = frame.loc[frame['state'] == state_name]
    if frame.empty:
        return {'asset': asset, 'macro_state': state_name, 'sample_count': 0, 'monthly_return': None, 'monthly_volatility': None, 'max_drawdown': None, 'win_rate': None, 'sample_start': None, 'sample_end': None}
    nav = (1.0 + frame['return']).cumprod()
    drawdown = nav / nav.cummax() - 1.0
    return {'asset': asset, 'macro_state': state_name, 'sample_count': int(len(frame)),
            'monthly_return': float(frame['return'].mean()),
            'monthly_volatility': float(frame['return'].std(ddof=0)),
            'max_drawdown': float(drawdown.min()),
            'win_rate': float((frame['return'] > 0).mean()),
            'sample_start': frame.index.min().date().isoformat(),
            'sample_end': frame.index.max().date().isoformat()}

rows = []
for asset in ASSETS:
    returns = asset_monthly_returns(asset)
    rows.extend(conditional_stats(asset, state, state_frame['rate_state'], returns) for state in ['加息/利率上行', '降息/利率下行'])
    rows.extend(conditional_stats(asset, state, state_frame['real_rate_state'], returns) for state in ['实际利率上行', '实际利率下行'])
condition_returns = pd.DataFrame(rows)
condition_returns.to_csv(DATA_DIR / 'condition_returns.csv', index=False, encoding='utf-8-sig')
baseline_rows = [
    {'asset': asset, 'baseline_monthly_return': float(asset_monthly_returns(asset).mean()), 'baseline_sample_count': int(len(asset_monthly_returns(asset)))}
    for asset in ASSETS
]
pd.DataFrame(baseline_rows).to_csv(DATA_DIR / 'baseline_returns.csv', index=False, encoding='utf-8-sig')
display_columns = {'asset': '资产', 'macro_state': '宏观状态', 'sample_count': '样本数', 'monthly_return': '月均收益', 'monthly_volatility': '月均波动', 'max_drawdown': '最大回撤', 'win_rate': '胜率'}
condition_returns.rename(columns=display_columns)

,资产,宏观状态,样本数,月均收益,月均波动,最大回撤,胜率,sample_start,sample_end
0,SPY,加息/利率上行,52,0.009717,0.046030,-0.288331,0.615385,2003-06-30,2026-07-31
1,SPY,降息/利率下行,41,-0.013613,0.047321,-0.640659,0.365854,2003-05-31,2026-02-28
2,SPY,实际利率上行,85,-0.002274,0.047067,-0.510777,0.552941,2003-03-31,2026-07-31
3,SPY,实际利率下行,88,0.010614,0.043941,-0.322133,0.625000,2003-02-28,2026-02-28
4,TLT,加息/利率上行,52,-0.048268,0.021969,-0.923349,0.000000,2003-06-30,2026-07-31
5,TLT,降息/利率下行,41,0.065699,0.028907,0.000000,1.000000,2003-05-31,2026-02-28
6,TLT,实际利率上行,85,-0.027609,0.027434,-0.910222,0.164706,2003-03-31,2026-07-31
7,TLT,实际利率下行,88,0.033346,0.038201,-0.130708,0.863636,2003-02-28,2026-02-28
8,GLD,加息/利率上行,47,-0.006012,0.049442,-0.495191,0.361702,2005-07-31,2026-07-31
9,GLD,降息/利率下行,37,0.031298,0.053633,-0.231127,0.729730,2004-12-31,2026-02-28


## 五档离散 modifier 建议

按「条件月均收益 − 全期基准月均收益」的偏差（百分点）离散分档，**仅供人工参考，不自动生效**：

| 偏差(pp) | modifier | 含义 |
|---|---|---|
| ≥ +1.5 | 1.15 | 强烈利好 |
| ≥ +0.5 且 < +1.5 | 1.08 | 温和利好 |
| > −0.5 且 < +0.5 | 1.00 | 中性 |
| ≤ −0.5 且 > −1.5 | 0.92 | 温和利空 |
| ≤ −1.5 | 0.85 | 强烈利空 |

样本联动：<24 个月仅作参考（REFERENCE_ONLY）；24–59 为候选（CANDIDATE）；≥60 才允许 APPROVED。
生效的 modifier 只能来自数据库中人工确认的 APPROVED 规则；无数据状态（期限结构/利率平稳等）为 1.00 保守中性，不落库。

In [5]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'qteasy_research').exists() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from qteasy_research.core.global_etf_engine import suggest_modifier_from_condition_returns

STATE_MAP = {
    '加息/利率上行': 'rate_up', '降息/利率下行': 'rate_down', '利率平稳': 'rate_stable',
    '期限结构倒挂': 'curve_inverted', '期限结构正常': 'curve_normal',
    '实际利率上行': 'real_yield_up', '实际利率下行': 'real_yield_down', '实际利率平稳': 'real_yield_stable',
}
baseline_returns = pd.read_csv(DATA_DIR / 'baseline_returns.csv')
suggest_rows = []
for _, row in condition_returns.iterrows():
    if row['sample_count'] <= 0:
        continue
    base = baseline_returns.loc[baseline_returns['asset'] == row['asset'], 'baseline_monthly_return'].iloc[0]
    s = suggest_modifier_from_condition_returns(
        condition_return=float(row['monthly_return']) * 100,
        baseline_return=float(base) * 100,
        sample_count=int(row['sample_count']),
    )
    suggest_rows.append({
        '资产': row['asset'], '宏观状态': row['macro_state'], '引擎状态': STATE_MAP[row['macro_state']],
        '样本数': int(row['sample_count']),
        '月均收益%': round(float(row['monthly_return']) * 100, 2),
        '全期基准%': round(float(base) * 100, 2),
        '偏差pp': round((float(row['monthly_return']) - float(base)) * 100, 2),
        '建议modifier': s['modifier'], '置信度': s['confidence'], '候选等级': s['status'],
    })
suggest_table = pd.DataFrame(suggest_rows)
suggest_table

,资产,宏观状态,引擎状态,样本数,月均收益%,全期基准%,偏差pp,建议modifier,置信度,候选等级
0,SPY,加息/利率上行,rate_up,52,0.97,1.01,-0.04,1.00,low,NEUTRAL
1,SPY,降息/利率下行,rate_down,41,-1.36,1.01,-2.37,0.85,high,CANDIDATE
2,SPY,实际利率上行,real_yield_up,85,-0.23,1.01,-1.24,0.92,medium,APPROVED
3,SPY,实际利率下行,real_yield_down,88,1.06,1.01,0.05,1.00,low,NEUTRAL
4,TLT,加息/利率上行,rate_up,52,-4.83,0.33,-5.16,0.85,high,CANDIDATE
5,TLT,降息/利率下行,rate_down,41,6.57,0.33,6.24,1.15,high,CANDIDATE
6,TLT,实际利率上行,real_yield_up,85,-2.76,0.33,-3.10,0.85,high,APPROVED
7,TLT,实际利率下行,real_yield_down,88,3.33,0.33,3.00,1.15,high,APPROVED
8,GLD,加息/利率上行,rate_up,47,-0.60,0.94,-1.54,0.85,high,CANDIDATE
9,GLD,降息/利率下行,rate_down,37,3.13,0.94,2.19,1.15,high,CANDIDATE


In [6]:
if state_frame.empty:
    print('无宏观数据，无法识别当前状态。')
else:
    current_states = [
        STATE_MAP[state_frame['rate_state'].iloc[-1]],
        STATE_MAP[state_frame['term_state'].iloc[-1]],
        STATE_MAP[state_frame['real_rate_state'].iloc[-1]],
    ]
    print('当前宏观状态三元组：', ' + '.join(current_states))
    print()
    print('候选规则后果：')
    print('- 利率状态样本 <60 个月 → 仅 CANDIDATE，不能 APPROVED；')
    print('- 期限结构状态无条件收益数据 → 保守中性，不落库；')
    print('- 实际利率状态样本 ≥60 个月 → 可 APPROVED。')
    print()
    print('三元组中任一分量缺少 APPROVED 规则时，引擎返回 PARTIAL（base_score 可算，宏观修正为空），属预期安全行为。')

当前宏观状态三元组： rate_up + curve_normal + real_yield_up

候选规则后果：
- 利率状态样本 <60 个月 → 仅 CANDIDATE，不能 APPROVED；
- 期限结构状态无条件收益数据 → 保守中性，不落库；
- 实际利率状态样本 ≥60 个月 → 可 APPROVED。

三元组中任一分量缺少 APPROVED 规则时，引擎返回 PARTIAL（base_score 可算，宏观修正为空），属预期安全行为。


## 研究结论边界

结论只能基于条件收益表；样本不足 24 个月的状态仅作参考，不进入正式宏观修正。

In [7]:
available = condition_returns.loc[condition_returns['sample_count'] > 0].copy()
print('利率回归代理：', rate_proxy)
print('有效条件结果行数：', len(available))
if available.empty:
    print('当前没有足够的本地数据，请先运行数据下载脚本。')
else:
    print('请人工比较各资产在不同宏观状态下的月均收益、最大回撤和胜率。')
    print('样本数少于 24 个月的状态仅作参考，不进入正式宏观修正。')

利率回归代理： DGS30
有效条件结果行数： 12
请人工比较各资产在不同宏观状态下的月均收益、最大回撤和胜率。
样本数少于 24 个月的状态仅作参考，不进入正式宏观修正。
